Import libraries

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import re
import unicodedata
import time

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report
)

print("Libraries loaded.")

Libraries loaded.


Upload files

In [2]:
from google.colab import files

uploaded = files.upload()

Saving test_set.csv to test_set.csv


In [3]:
from google.colab import files

uploaded = files.upload()

Saving single_obfuscation_attacks.csv to single_obfuscation_attacks.csv


In [4]:
from google.colab import files

uploaded = files.upload()

Saving multi_obfuscation_attacks.csv to multi_obfuscation_attacks.csv


In [5]:
from google.colab import files

uploaded = files.upload()

Saving hurtlex_EN.tsv to hurtlex_EN.tsv


Load datasets

In [6]:
clean_test = pd.read_csv("test_set.csv")

single_test = pd.read_csv(
    "single_obfuscation_attacks.csv"
)

multi_test = pd.read_csv(
    "multi_obfuscation_attacks.csv"
)

hurtlex = pd.read_csv(
    "hurtlex_EN.tsv",
    sep="\t"
)

print("Clean test:", clean_test.shape)
print("Single obfuscation:", single_test.shape)
print("Multi obfuscation:", multi_test.shape)
print("HurtLex:", hurtlex.shape)

Clean test: (22777, 5)
Single obfuscation: (42749, 10)
Multi obfuscation: (7872, 11)
HurtLex: (8228, 6)


In [7]:
# Inspect columns

print("CLEAN TEST COLUMNS")
print(clean_test.columns.tolist())

print("\nSINGLE OBFUSCATION COLUMNS")
print(single_test.columns.tolist())

print("\nMULTI OBFUSCATION COLUMNS")
print(multi_test.columns.tolist())

print("\nHURTLEX COLUMNS")
print(hurtlex.columns.tolist())

CLEAN TEST COLUMNS
['id', 'text', 'text_length', 'source_dataset', 'abusive']

SINGLE OBFUSCATION COLUMNS
['id', 'text', 'text_length', 'source_dataset', 'abusive', 'original_text', 'obfuscated_text', 'obfuscation_type', 'severity', 'obfuscated_length']

MULTI OBFUSCATION COLUMNS
['id', 'text', 'text_length', 'source_dataset', 'abusive', 'original_text', 'obfuscated_text', 'obfuscation_type', 'attack_count', 'severity', 'obfuscated_length']

HURTLEX COLUMNS
['id', 'pos', 'category', 'stereotype', 'lemma', 'level']


In [8]:
# Basic cleaning

clean_test["text"] = (
    clean_test["text"]
    .fillna("")
    .astype(str)
)

clean_test["abusive"] = (
    clean_test["abusive"]
    .astype(str)
    .str.strip()
    .str.upper()
)

print(clean_test["abusive"].value_counts())

abusive
NOT    20054
OFF     2723
Name: count, dtype: int64


Hurtlex keyword baseline

In [9]:
hurtlex["level"] = (
    hurtlex["level"]
    .astype(str)
    .str.lower()
    .str.strip()
)

hurtlex_conservative = hurtlex[
    hurtlex["level"] == "conservative"
].copy()

print(
    "Conservative HurtLex entries:",
    len(hurtlex_conservative)
)

Conservative HurtLex entries: 3360


In [10]:
# Extract keyword list

keywords = (
    hurtlex_conservative["lemma"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

keywords = [
    k
    for k in keywords
    if k != ""
]

keywords = sorted(
    set(keywords),
    key=len,
    reverse=True
)

print(
    "Unique HurtLex keywords:",
    len(keywords)
)

print(
    "Example keywords:",
    keywords[:20]
)

Unique HurtLex keywords: 2227
Example keywords: ['controversies concerning the word "niggardly"', 'controversies about the word "niggardly"', 'persons involved in the sex trade', 'out-of-control sexual behaviours', 'out of control sexual behaviours', 'out-of-control sexual behaviors', 'out of control sexual behaviors', 'out-of-control sexual behaviour', 'out of control sexual behaviour', 'out of control sexual behavior', 'out-of-control sexual behavior', 'society for individual rights', 'paraphilia related disorders', 'etymology of the word "fuck"', 'f%e2%80%93%e2%80%93%e2%80%93', 'paraphilia-related disorders', 'compulsive sexual behaviours', 'paraphiliarelated disorders', 'behavioral signals of lying', 'compulsive sexual behaviour']


In [11]:
keyword_regex = re.compile(
    r"(?<!\w)(?:"
    + "|".join(
        re.escape(keyword)
        for keyword in keywords
    )
    + r")(?!\w)",
    flags=re.IGNORECASE
)

print("Keyword regex compiled.")

Keyword regex compiled.


In [12]:
def keyword_predict(text):

    if pd.isna(text):
        return "NOT"

    text = str(text).lower()

    if keyword_regex.search(text):
        return "OFF"

    return "NOT"

In [13]:
# Test keyword baseline

sample = clean_test.sample(
    min(10, len(clean_test)),
    random_state=42
).copy()

sample["keyword_prediction"] = (
    sample["text"]
    .apply(keyword_predict)
)

display(
    sample[
        [
            "text",
            "abusive",
            "keyword_prediction"
        ]
    ]
)

,text,abusive,keyword_prediction
1595,Kira Speer loves volleyball. She spikes it har...,NOT,NOT
10902,"this article does read like an ad, and without...",NOT,NOT
4699,any one noticing recent surge of refugees from...,NOT,NOT
17916,Hi Tim. The latest press release is from PR N...,NOT,NOT
15992,Please stop your disruptive editing. \n\nPleas...,NOT,NOT
18957,::::::You've been topic banned from areas of p...,NOT,NOT
11712,== Not an urban legend == \n\n He actually sai...,NOT,OFF
13836,"""Oh, and it is good that you've selected readi...",NOT,NOT
15066,""" \n This article contradicts itself. It stat...",NOT,NOT
20346,MegaAward! Nomination \nYou have been nominate...,NOT,OFF


Rule-based normalisation

In [14]:
# Character mappings

CONFUSABLE_MAP = str.maketrans({

    # Cyrillic → Latin
    "а": "a",
    "е": "e",
    "о": "o",
    "р": "p",
    "с": "c",
    "х": "x",
    "у": "y",
    "і": "i",

    # Common Greek confusables
    "Α": "a",
    "Β": "b",
    "Ε": "e",
    "Ι": "i",
    "Κ": "k",
    "Μ": "m",
    "Ν": "n",
    "Ο": "o",
    "Ρ": "p",
    "Τ": "t",
    "Χ": "x",
})


LEET_MAP = str.maketrans({

    "0": "o",
    "1": "i",
    "3": "e",
    "4": "a",
    "5": "s",
    "7": "t",
    "@": "a",
    "$": "s",
})

In [15]:
# Normalisation function

def normalise_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode compatibility normalisation
    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # Lowercase
    text = text.lower()

    # Remove zero-width/invisible characters
    text = re.sub(
        r"[\u200b\u200c\u200d\ufeff]",
        "",
        text
    )

    # Convert Unicode confusables
    text = text.translate(
        CONFUSABLE_MAP
    )

    # Convert leetspeak / substitutions
    text = text.translate(
        LEET_MAP
    )

    # Collapse excessive repeated characters
    # Example: "stuuuupid" -> "stupid"
    text = re.sub(
        r"(.)\1{2,}",
        r"\1",
        text
    )

    # Standardise whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [16]:
# Normalise HurtLex once

normalised_keywords = []

for keyword in keywords:

    normalised = normalise_text(
        keyword
    )

    if normalised:
        normalised_keywords.append(
            normalised
        )

normalised_keywords = sorted(
    set(normalised_keywords),
    key=len,
    reverse=True
)

print(
    "Normalised HurtLex keywords:",
    len(normalised_keywords)
)

Normalised HurtLex keywords: 2226


In [17]:
normalised_keyword_regex = re.compile(
    r"(?<!\w)(?:"
    + "|".join(
        re.escape(keyword)
        for keyword
        in normalised_keywords
    )
    + r")(?!\w)",
    flags=re.IGNORECASE
)

print(
    "Normalised keyword regex compiled."
)

Normalised keyword regex compiled.


In [18]:
# Compact HurtLex terms

compact_keywords = []

for keyword in normalised_keywords:

    compact = re.sub(
        r"[^a-z0-9]",
        "",
        keyword
    )

    # Minimum length helps reduce false positives
    if len(compact) >= 4:
        compact_keywords.append(
            compact
        )

compact_keywords = sorted(
    set(compact_keywords),
    key=len,
    reverse=True
)

print(
    "Compact keywords:",
    len(compact_keywords)
)

Compact keywords: 2021


In [19]:
compact_keyword_regex = re.compile(
    "|".join(
        re.escape(keyword)
        for keyword
        in compact_keywords
    )
)

print(
    "Compact keyword regex compiled."
)

Compact keyword regex compiled.


In [20]:
# Fast rule detector for CLEAN text

def normalised_keyword_predict_clean(text):

    if pd.isna(text):
        return "NOT"

    normalised = normalise_text(
        text
    )

    if normalised_keyword_regex.search(
        normalised
    ):
        return "OFF"

    return "NOT"

In [21]:
# Rule detector for OBFUSCATED text

def normalised_keyword_predict_attack(text):

    if pd.isna(text):
        return "NOT"

    # Normalise
    normalised = normalise_text(
        text
    )

    # First attempt:
    # normal word matching
    if normalised_keyword_regex.search(
        normalised
    ):
        return "OFF"

    # Second attempt:
    # remove symbols/spaces
    compact = re.sub(
        r"[^a-z0-9]",
        "",
        normalised
    )

    if compact_keyword_regex.search(
        compact
    ):
        return "OFF"

    return "NOT"

Evaluation

In [22]:
def evaluate(y_true, y_pred):

    precision = precision_score(
        y_true,
        y_pred,
        pos_label="OFF",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label="OFF",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label="OFF",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    return {

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "accuracy": accuracy
    }

In [23]:
# CLEAN keyword baseline

start = time.time()

clean_keyword_pred = (
    clean_test["text"]
    .apply(keyword_predict)
)

clean_keyword_metrics = evaluate(
    clean_test["abusive"],
    clean_keyword_pred
)

end = time.time()

print(
    "KEYWORD BASELINE — CLEAN"
)

print(
    clean_keyword_metrics
)

print(
    f"Runtime: {end-start:.2f} seconds"
)

KEYWORD BASELINE — CLEAN
{'precision': 0.22353836122582046, 'recall': 0.7554168196841718, 'f1': 0.3449895178197065, 'accuracy': 0.6570663388505948}
Runtime: 20.87 seconds


In [24]:
# CLEAN rule-based model

start = time.time()

clean_rule_pred = (
    clean_test["text"]
    .apply(
        normalised_keyword_predict_clean
    )
)

clean_rule_metrics = evaluate(
    clean_test["abusive"],
    clean_rule_pred
)

end = time.time()

print(
    "RULE-BASED — CLEAN"
)

print(
    clean_rule_metrics
)

print(
    f"Runtime: {end-start:.2f} seconds"
)

RULE-BASED — CLEAN
{'precision': 0.2234527687296417, 'recall': 0.7557840616966581, 'f1': 0.3449258359172044, 'accuracy': 0.6568029152214954}
Runtime: 24.08 seconds


In [25]:
# Clean comparison

clean_results = pd.DataFrame([

    {
        "model": "keyword",
        "condition": "clean",
        "precision":
            clean_keyword_metrics["precision"],
        "recall":
            clean_keyword_metrics["recall"],
        "f1":
            clean_keyword_metrics["f1"],
        "accuracy":
            clean_keyword_metrics["accuracy"]
    },

    {
        "model": "rule_based",
        "condition": "clean",
        "precision":
            clean_rule_metrics["precision"],
        "recall":
            clean_rule_metrics["recall"],
        "f1":
            clean_rule_metrics["f1"],
        "accuracy":
            clean_rule_metrics["accuracy"]
    }

])

display(clean_results)

,model,condition,precision,recall,f1,accuracy
0,keyword,clean,0.223538,0.755417,0.344990,0.657066
1,rule_based,clean,0.223453,0.755784,0.344926,0.656803


In [26]:
# Robustness calculation

def calculate_robustness(
    clean_f1,
    attacked_f1
):

    absolute_drop = (
        clean_f1
        -
        attacked_f1
    )

    if clean_f1 > 0:

        relative_drop = (
            absolute_drop
            /
            clean_f1
        ) * 100

    else:

        relative_drop = 0

    return (
        absolute_drop,
        relative_drop
    )

In [27]:
clean_scores = {

    "keyword":
        clean_keyword_metrics["f1"],

    "rule_based":
        clean_rule_metrics["f1"]
}

print(clean_scores)

{'keyword': 0.3449895178197065, 'rule_based': 0.3449258359172044}


In [28]:
# Inspect attack dataset

display(
    single_test.head()
)

print(
    single_test.columns.tolist()
)

,id,text,text_length,source_dataset,abusive,original_text,obfuscated_text,obfuscation_type,severity,obfuscated_length
0,jigsaw_88344,Just stating the truth. I know how you people ...,245,JIGSAW,OFF,Just stating the truth. I know how you people ...,Just stating the truth. I know how you people ...,clean,none,245
1,jigsaw_88344,Just stating the truth. I know how you people ...,245,JIGSAW,OFF,Just stating the truth. I know how you people ...,Ju5t sta7ing the tru7h. I know h0w you p3opl3 ...,character_substitution,low,245
2,jigsaw_88344,Just stating the truth. I know how you people ...,245,JIGSAW,OFF,Just stating the truth. I know how you people ...,Ju5t st4ting th3 tru7h. ! know how y0u p3ople ...,character_substitution,medium,245
3,jigsaw_88344,Just stating the truth. I know how you people ...,245,JIGSAW,OFF,Just stating the truth. I know how you people ...,Jus7 st@7!ng 7h3 tru7h. I kn0w h0w you p3op13 ...,character_substitution,high,245
4,jigsaw_88344,Just stating the truth. I know how you people ...,245,JIGSAW,OFF,Just stating the truth. I know how you people ...,Just stating the truth. I know how yo u people...,spacing,low,261


['id', 'text', 'text_length', 'source_dataset', 'abusive', 'original_text', 'obfuscated_text', 'obfuscation_type', 'severity', 'obfuscated_length']


In [29]:
OBF_TEXT_COLUMN = "obfuscated_text"

In [30]:
if "abusive" in single_test.columns:

    single_test["abusive"] = (
        single_test["abusive"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

In [31]:
if "abusive" in multi_test.columns:

    multi_test["abusive"] = (
        multi_test["abusive"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

In [32]:
# Extract clean NOT examples

clean_not = clean_test[
    clean_test["abusive"] == "NOT"
].copy()

print(
    "Clean NOT samples:",
    len(clean_not)
)

Clean NOT samples: 20054


In [33]:
# Attack evaluation set

def make_attack_evaluation_set(
    attacked_rows
):

    attacked = attacked_rows.copy()

    # Ensure labels exist
    if "abusive" not in attacked.columns:

        attacked["abusive"] = "OFF"

    attacked[
        "evaluation_text"
    ] = attacked[
        OBF_TEXT_COLUMN
    ].fillna("").astype(str)

    # Obfuscated OFF examples
    attacked_eval = attacked[
        [
            "evaluation_text",
            "abusive"
        ]
    ].copy()

    # Unchanged NOT examples
    negatives = clean_not.copy()

    negatives[
        "evaluation_text"
    ] = negatives[
        "text"
    ]

    negatives_eval = negatives[
        [
            "evaluation_text",
            "abusive"
        ]
    ].copy()

    # Combine
    combined = pd.concat(
        [
            attacked_eval,
            negatives_eval
        ],
        ignore_index=True
    )

    return combined

In [34]:
print(
    single_test.columns.tolist()
)

['id', 'text', 'text_length', 'source_dataset', 'abusive', 'original_text', 'obfuscated_text', 'obfuscation_type', 'severity', 'obfuscated_length']


In [35]:
print(
    single_test[
        "obfuscation_type"
    ].value_counts()
)

print(
    single_test[
        "severity"
    ].value_counts()
)

obfuscation_type
character_substitution    8169
symbolic_masking          8169
unicode_confusable        8169
misspelling               8154
spacing                   7365
clean                     2723
Name: count, dtype: int64
severity
low       13473
medium    13360
high      13193
none       2723
Name: count, dtype: int64


In [36]:
# Single-obfuscation evaluation

single_results = []


for obf_type in (
    single_test[
        "obfuscation_type"
    ].dropna().unique()
):

    for severity in (
        single_test[
            "severity"
        ].dropna().unique()
    ):

        attacked = single_test[

            (
                single_test[
                    "obfuscation_type"
                ] == obf_type
            )

            &

            (
                single_test[
                    "severity"
                ] == severity
            )

        ].copy()


        if len(attacked) == 0:
            continue


        eval_df = (
            make_attack_evaluation_set(
                attacked
            )
        )


        # KEYWORD BASELINE

        keyword_preds = (
            eval_df[
                "evaluation_text"
            ]
            .apply(
                keyword_predict
            )
        )


        metrics = evaluate(
            eval_df["abusive"],
            keyword_preds
        )


        drop, relative_drop = (
            calculate_robustness(

                clean_scores[
                    "keyword"
                ],

                metrics["f1"]

            )
        )


        single_results.append({

            "model":
                "keyword",

            "attack_type":
                "single",

            "obfuscation_type":
                obf_type,

            "severity":
                severity,

            "n_samples":
                len(eval_df),

            "precision":
                metrics["precision"],

            "recall":
                metrics["recall"],

            "f1":
                metrics["f1"],

            "accuracy":
                metrics["accuracy"],

            "clean_f1":
                clean_scores[
                    "keyword"
                ],

            "robustness_drop":
                drop,

            "relative_drop_pct":
                relative_drop
        })


        # RULE-BASED SYSTEM

        rule_preds = (
            eval_df[
                "evaluation_text"
            ]
            .apply(
                normalised_keyword_predict_attack
            )
        )


        metrics = evaluate(
            eval_df["abusive"],
            rule_preds
        )


        drop, relative_drop = (
            calculate_robustness(

                clean_scores[
                    "rule_based"
                ],

                metrics["f1"]

            )
        )


        single_results.append({

            "model":
                "rule_based",

            "attack_type":
                "single",

            "obfuscation_type":
                obf_type,

            "severity":
                severity,

            "n_samples":
                len(eval_df),

            "precision":
                metrics["precision"],

            "recall":
                metrics["recall"],

            "f1":
                metrics["f1"],

            "accuracy":
                metrics["accuracy"],

            "clean_f1":
                clean_scores[
                    "rule_based"
                ],

            "robustness_drop":
                drop,

            "relative_drop_pct":
                relative_drop
        })


single_results_df = pd.DataFrame(
    single_results
)

display(
    single_results_df
)

,model,attack_type,obfuscation_type,severity,n_samples,precision,recall,f1,accuracy,clean_f1,robustness_drop,relative_drop_pct
0,keyword,single,clean,none,22777,0.223538,0.755417,0.344990,0.657066,0.344990,0.000000,0.000000
1,rule_based,single,clean,none,22777,0.173340,0.845758,0.287713,0.499363,0.344926,0.057213,16.586950
2,keyword,single,character_substitution,low,22777,0.189082,0.611825,0.288885,0.639900,0.344990,0.056104,16.262663
3,rule_based,single,character_substitution,low,22777,0.168018,0.814543,0.278573,0.495632,0.344926,0.066353,19.236777
4,keyword,single,character_substitution,medium,22777,0.144721,0.443996,0.218290,0.619836,0.344990,0.126699,36.725570
5,rule_based,single,character_substitution,medium,22777,0.159936,0.767903,0.264734,0.490056,0.344926,0.080192,23.249063
6,keyword,single,character_substitution,high,22777,0.098309,0.286082,0.146332,0.600957,0.344990,0.198657,57.583552
7,rule_based,single,character_substitution,high,22777,0.150974,0.717224,0.249441,0.483997,0.344926,0.095485,27.682652
8,keyword,single,spacing,low,22639,0.174084,0.582592,0.268067,0.636733,0.344990,0.076923,22.297080
9,rule_based,single,spacing,low,22639,0.164282,0.835203,0.274560,0.496047,0.344926,0.070366,20.400374


In [37]:
single_results_df = (
    single_results_df
    .sort_values(
        [
            "model",
            "obfuscation_type",
            "severity"
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    single_results_df
)

,model,attack_type,obfuscation_type,severity,n_samples,precision,recall,f1,accuracy,clean_f1,robustness_drop,relative_drop_pct
0,keyword,single,character_substitution,high,22777,0.098309,0.286082,0.146332,0.600957,0.344990,0.198657,57.583552
1,keyword,single,character_substitution,low,22777,0.189082,0.611825,0.288885,0.639900,0.344990,0.056104,16.262663
2,keyword,single,character_substitution,medium,22777,0.144721,0.443996,0.218290,0.619836,0.344990,0.126699,36.725570
3,keyword,single,clean,none,22777,0.223538,0.755417,0.344990,0.657066,0.344990,0.000000,0.000000
4,keyword,single,misspelling,high,22771,0.161582,0.506809,0.245040,0.627377,0.344990,0.099950,28.971873
5,keyword,single,misspelling,low,22773,0.211456,0.704671,0.325297,0.650990,0.344990,0.019692,5.708117
6,keyword,single,misspelling,medium,22772,0.197191,0.645695,0.302117,0.643949,0.344990,0.042872,12.427077
7,keyword,single,spacing,high,22361,0.056267,0.184655,0.086252,0.596351,0.344990,0.258737,74.998580
8,keyword,single,spacing,low,22639,0.174084,0.582592,0.268067,0.636733,0.344990,0.076923,22.297080
9,keyword,single,spacing,medium,22527,0.115061,0.375657,0.176164,0.614285,0.344990,0.168826,48.936466


In [38]:
print(
    multi_test.columns.tolist()
)

['id', 'text', 'text_length', 'source_dataset', 'abusive', 'original_text', 'obfuscated_text', 'obfuscation_type', 'attack_count', 'severity', 'obfuscated_length']


In [39]:
# Multi-obfuscation evaluation

multi_results = []

# Decide how to group the multi-obfuscation data
if "severity" in multi_test.columns:
    group_column = "severity"

elif "attack_count" in multi_test.columns:
    group_column = "attack_count"

elif "obfuscation_type" in multi_test.columns:
    group_column = "obfuscation_type"

else:
    group_column = None


print("Multi-obfuscation grouping column:", group_column)

Multi-obfuscation grouping column: severity


In [40]:
# Evaluate multi-obfuscation conditions

if group_column is not None:

    groups = multi_test[group_column].dropna().unique()

else:

    # Treat entire multi-obfuscation dataset as one condition
    groups = ["all_multi"]


for group_value in groups:

    # Select attack rows
    if group_column is not None:

        attacked = multi_test[
            multi_test[group_column] == group_value
        ].copy()

    else:

        attacked = multi_test.copy()


    if len(attacked) == 0:
        continue


    # Create evaluation set: obfuscated OFF + unchanged NOT
    eval_df = make_attack_evaluation_set(
        attacked
    )


    # KEYWORD BASELINE

    keyword_preds = (
        eval_df["evaluation_text"]
        .apply(keyword_predict)
    )


    keyword_metrics = evaluate(
        eval_df["abusive"],
        keyword_preds
    )


    keyword_drop, keyword_relative_drop = (
        calculate_robustness(
            clean_scores["keyword"],
            keyword_metrics["f1"]
        )
    )


    multi_results.append({

        "model": "keyword",

        "attack_type": "multi",

        "group_column": group_column
            if group_column is not None
            else "all",

        "group_value": group_value,

        "n_samples": len(eval_df),

        "precision":
            keyword_metrics["precision"],

        "recall":
            keyword_metrics["recall"],

        "f1":
            keyword_metrics["f1"],

        "accuracy":
            keyword_metrics["accuracy"],

        "clean_f1":
            clean_scores["keyword"],

        "robustness_drop":
            keyword_drop,

        "relative_drop_pct":
            keyword_relative_drop
    })


    # RULE-BASED NORMALISATION

    rule_preds = (
        eval_df["evaluation_text"]
        .apply(
            normalised_keyword_predict_attack
        )
    )


    rule_metrics = evaluate(
        eval_df["abusive"],
        rule_preds
    )


    rule_drop, rule_relative_drop = (
        calculate_robustness(
            clean_scores["rule_based"],
            rule_metrics["f1"]
        )
    )


    multi_results.append({

        "model": "rule_based",

        "attack_type": "multi",

        "group_column": group_column
            if group_column is not None
            else "all",

        "group_value": group_value,

        "n_samples": len(eval_df),

        "precision":
            rule_metrics["precision"],

        "recall":
            rule_metrics["recall"],

        "f1":
            rule_metrics["f1"],

        "accuracy":
            rule_metrics["accuracy"],

        "clean_f1":
            clean_scores["rule_based"],

        "robustness_drop":
            rule_drop,

        "relative_drop_pct":
            rule_relative_drop
    })


# Convert results to DataFrame
multi_results_df = pd.DataFrame(
    multi_results
)

display(multi_results_df)

,model,attack_type,group_column,group_value,n_samples,precision,recall,f1,accuracy,clean_f1,robustness_drop,relative_drop_pct
0,keyword,multi,severity,low,22726,0.154438,0.488398,0.234670,0.625451,0.344990,0.110319,31.977637
1,rule_based,multi,severity,low,22726,0.149659,0.723428,0.248011,0.484203,0.344926,0.096915,28.097213
2,keyword,multi,severity,medium,22682,0.079845,0.235921,0.119311,0.596464,0.344990,0.225678,65.416029
3,rule_based,multi,severity,medium,22682,0.123953,0.591324,0.204946,0.468433,0.344926,0.139980,40.582706
4,keyword,multi,severity,high,22626,0.029739,0.085148,0.044082,0.580217,0.344990,0.300907,87.222184
5,rule_based,multi,severity,high,22626,0.090058,0.422628,0.148477,0.448953,0.344926,0.196449,56.953939


In [41]:
# Clean rows for master table

clean_master = pd.DataFrame([

    {

        "model":
            "keyword",

        "attack_type":
            "clean",

        "obfuscation_type":
            "none",

        "severity":
            "none",

        "techniques":
            "none",

        "n_samples":
            len(clean_test),

        "precision":
            clean_keyword_metrics[
                "precision"
            ],

        "recall":
            clean_keyword_metrics[
                "recall"
            ],

        "f1":
            clean_keyword_metrics[
                "f1"
            ],

        "accuracy":
            clean_keyword_metrics[
                "accuracy"
            ],

        "clean_f1":
            clean_keyword_metrics[
                "f1"
            ],

        "robustness_drop":
            0,

        "relative_drop_pct":
            0
    },


    {

        "model":
            "rule_based",

        "attack_type":
            "clean",

        "obfuscation_type":
            "none",

        "severity":
            "none",

        "techniques":
            "none",

        "n_samples":
            len(clean_test),

        "precision":
            clean_rule_metrics[
                "precision"
            ],

        "recall":
            clean_rule_metrics[
                "recall"
            ],

        "f1":
            clean_rule_metrics[
                "f1"
            ],

        "accuracy":
            clean_rule_metrics[
                "accuracy"
            ],

        "clean_f1":
            clean_rule_metrics[
                "f1"
            ],

        "robustness_drop":
            0,

        "relative_drop_pct":
            0
    }

])

In [42]:
# Master results

master_results = pd.concat(
    [
        clean_master,
        single_results_df,
        multi_results_df
    ],
    ignore_index=True,
    sort=False
)

display(
    master_results
)

,model,attack_type,obfuscation_type,severity,techniques,n_samples,precision,recall,f1,accuracy,clean_f1,robustness_drop,relative_drop_pct,group_column,group_value
0,keyword,clean,none,none,none,22777,0.223538,0.755417,0.344990,0.657066,0.344990,0.000000,0.000000,NaN,NaN
1,rule_based,clean,none,none,none,22777,0.223453,0.755784,0.344926,0.656803,0.344926,0.000000,0.000000,NaN,NaN
2,keyword,single,character_substitution,high,NaN,22777,0.098309,0.286082,0.146332,0.600957,0.344990,0.198657,57.583552,NaN,NaN
3,keyword,single,character_substitution,low,NaN,22777,0.189082,0.611825,0.288885,0.639900,0.344990,0.056104,16.262663,NaN,NaN
4,keyword,single,character_substitution,medium,NaN,22777,0.144721,0.443996,0.218290,0.619836,0.344990,0.126699,36.725570,NaN,NaN
5,keyword,single,clean,none,NaN,22777,0.223538,0.755417,0.344990,0.657066,0.344990,0.000000,0.000000,NaN,NaN
6,keyword,single,misspelling,high,NaN,22771,0.161582,0.506809,0.245040,0.627377,0.344990,0.099950,28.971873,NaN,NaN
7,keyword,single,misspelling,low,NaN,22773,0.211456,0.704671,0.325297,0.650990,0.344990,0.019692,5.708117,NaN,NaN
8,keyword,single,misspelling,medium,NaN,22772,0.197191,0.645695,0.302117,0.643949,0.344990,0.042872,12.427077,NaN,NaN
9,keyword,single,spacing,high,NaN,22361,0.056267,0.184655,0.086252,0.596351,0.344990,0.258737,74.998580,NaN,NaN


In [43]:
# Round metrics

metric_columns = [

    "precision",
    "recall",
    "f1",
    "accuracy",
    "clean_f1",
    "robustness_drop",
    "relative_drop_pct"
]


for column in metric_columns:

    if column in master_results.columns:

        master_results[column] = (
            master_results[column]
            .round(4)
        )


display(
    master_results
)

,model,attack_type,obfuscation_type,severity,techniques,n_samples,precision,recall,f1,accuracy,clean_f1,robustness_drop,relative_drop_pct,group_column,group_value
0,keyword,clean,none,none,none,22777,0.2235,0.7554,0.3450,0.6571,0.3450,0.0000,0.0000,NaN,NaN
1,rule_based,clean,none,none,none,22777,0.2235,0.7558,0.3449,0.6568,0.3449,0.0000,0.0000,NaN,NaN
2,keyword,single,character_substitution,high,NaN,22777,0.0983,0.2861,0.1463,0.6010,0.3450,0.1987,57.5836,NaN,NaN
3,keyword,single,character_substitution,low,NaN,22777,0.1891,0.6118,0.2889,0.6399,0.3450,0.0561,16.2627,NaN,NaN
4,keyword,single,character_substitution,medium,NaN,22777,0.1447,0.4440,0.2183,0.6198,0.3450,0.1267,36.7256,NaN,NaN
5,keyword,single,clean,none,NaN,22777,0.2235,0.7554,0.3450,0.6571,0.3450,0.0000,0.0000,NaN,NaN
6,keyword,single,misspelling,high,NaN,22771,0.1616,0.5068,0.2450,0.6274,0.3450,0.0999,28.9719,NaN,NaN
7,keyword,single,misspelling,low,NaN,22773,0.2115,0.7047,0.3253,0.6510,0.3450,0.0197,5.7081,NaN,NaN
8,keyword,single,misspelling,medium,NaN,22772,0.1972,0.6457,0.3021,0.6439,0.3450,0.0429,12.4271,NaN,NaN
9,keyword,single,spacing,high,NaN,22361,0.0563,0.1847,0.0863,0.5964,0.3450,0.2587,74.9986,NaN,NaN


In [44]:
master_results.to_csv(
    "baseline_rule_based_master_results.csv",
    index=False
)

single_results_df.to_csv(
    "baseline_single_results.csv",
    index=False
)

multi_results_df.to_csv(
    "baseline_multi_results.csv",
    index=False
)

print(
    "Results saved successfully."
)

Results saved successfully.


In [45]:
clean_predictions = clean_test.copy()

clean_predictions[
    "keyword_prediction"
] = clean_keyword_pred

clean_predictions[
    "rule_prediction"
] = clean_rule_pred

clean_predictions[
    "keyword_correct"
] = (

    clean_predictions[
        "keyword_prediction"
    ]

    ==

    clean_predictions[
        "abusive"
    ]
)

clean_predictions[
    "rule_correct"
] = (

    clean_predictions[
        "rule_prediction"
    ]

    ==

    clean_predictions[
        "abusive"
    ]
)

clean_predictions.to_csv(
    "clean_baseline_predictions.csv",
    index=False
)

display(
    clean_predictions.head()
)

,id,text,text_length,source_dataset,abusive,keyword_prediction,rule_prediction,keyword_correct,rule_correct
0,hatexplain_684,i am half black and i grew up muslim,36,HateXplain,NOT,OFF,OFF,False,False
1,jigsaw_133950,"No, it really isn't. I'll remove it now... -",44,JIGSAW,NOT,NOT,NOT,True,True
2,jigsaw_61896,"(Thanks, N.h., for catching the header that I ...",54,JIGSAW,NOT,NOT,NOT,True,True
3,jigsaw_25578,"""\nThis is not a reason to speedily delete it....",225,JIGSAW,NOT,NOT,NOT,True,True
4,jigsaw_114641,== Sorry == \n\n I am sorry Sarah...:(,36,JIGSAW,NOT,NOT,NOT,True,True


In [46]:
from google.colab import files

files.download(
    "baseline_rule_based_master_results.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>